# `decomposer` (DecomposerModule)
- **Category**: Logic (LLM)
- **Role**: 복합 자연어 질문을 각 시점 및 계정 항목 단위의 정형 서브쿼리(`Company: ... | Sheet: ... | Row Header: ... | Column Header: ... | Cell Value: ?`)로 분해합니다.


In [ ]:
import sys
from pathlib import Path
import json

# 프로젝트 루트 경로 등록
PROJECT_ROOT = Path(".").resolve().parent.parent if Path(".").resolve().name == "modules" else Path(".").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

def print_io(title: str, input_data: dict, output_data: dict):
    print("=" * 70)
    print(f"📌 [Module Execution] {title}")
    print("=" * 70)
    print("\n📥 [Input DTO]")
    print(json.dumps(input_data, indent=2, ensure_ascii=False))
    print("\n📤 [Output Result]")
    print(json.dumps(output_data, indent=2, ensure_ascii=False))
    print("\n")


In [ ]:
from unittest.mock import MagicMock
from modules.query.decomposer import DecomposerModule, DecomposerInputDTO, DecomposerConfigDTO
from backend.providers.llm.chat_completion import ChatCompletionResult

# Mock LLM Client (OpenAI API 키 없이도 결정론적 테스트 가능)
mock_llm = MagicMock()
mock_llm.complete_with_metadata.return_value = ChatCompletionResult(
    content=json.dumps({
        "items": [
            {
                "company": "삼성전자",
                "sheet": "손익계산서",
                "row_header": "영업이익",
                "column_header": "2022",
                "cell_value": "?"
            },
            {
                "company": "삼성전자",
                "sheet": "손익계산서",
                "row_header": "영업이익",
                "column_header": "2023",
                "cell_value": "?"
            }
        ]
    }),
    usage={"prompt_tokens": 45, "completion_tokens": 60, "total_tokens": 105},
    latency_seconds=0.35,
)

module = DecomposerModule(completion_client=mock_llm)

sample_input = {
    "query_context": {
        "question_id": "QUERY-001",
        "question_text": "2023년 삼성전자 영업이익과 2022년 대비 증감율은 얼마인가요?"
    }
}
input_dto = DecomposerInputDTO(**sample_input)
output = module.run(input_dto, config=DecomposerConfigDTO(model="gpt-5.6-luna"))
print_io("decomposer (DecomposerModule)", sample_input, output)
